In [2]:
# imports e configs
import os, random
import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from segmentacao import segmentar_grao, desenhar_contorno

# Pasta raiz com imagens (todas as classes em subpastas dentro de `archive`)
DATA_DIR = Path('archive')
# lista ordenada de subpastas (uma por classe)
class_dirs = sorted([p for p in DATA_DIR.iterdir() if p.is_dir()]) if DATA_DIR.exists() else []
if len(class_dirs) >= 2:
    dark_dir = class_dirs[0]
    medium_dir = class_dirs[1]
else:
    # fallback: se não houver subpastas, utiliza a própria pasta (pode precisar ajustar)
    dark_dir = DATA_DIR
    medium_dir = DATA_DIR

plt.rcParams['figure.dpi'] = 120
random.seed(42)
np.random.seed(42)


In [6]:
# Funções auxiliares para EDA (nomes em português)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


def carregar_features_csv(caminho='outputs/features.csv'):
    df = pd.read_csv(caminho)
    return df


def plot_boxplots_por_rotulo(df, features, caminho_saida='outputs/boxplots_custom.png'):
    fig, axes = plt.subplots(1, len(features), figsize=(4*len(features), 4))
    if len(features) == 1:
        axes = [axes]
    for i, f in enumerate(features):
        try:
            df.boxplot(column=f, by='rotulo', ax=axes[i])
            axes[i].set_title(f)
        except Exception as e:
            axes[i].text(0.5, 0.5, f'Erro em {f}: {e}', ha='center')
    plt.suptitle('')
    plt.tight_layout()
    plt.savefig(caminho_saida)
    plt.close()


def plot_pca_2d(df, caminho_saida='outputs/pca_2d.png'):
    X = df.drop(columns=['rotulo','caminho']).values
    X = np.nan_to_num(X)
    y = df['rotulo'].values
    pca = PCA(n_components=2, random_state=42)
    Z = pca.fit_transform(X)
    plt.figure(figsize=(7,6))
    for rotulo in np.unique(y):
        sel = y == rotulo
        plt.scatter(Z[sel,0], Z[sel,1], label=rotulo, alpha=0.7)
    plt.legend()
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.title('PCA 2D – projeção')
    plt.tight_layout()
    plt.savefig(caminho_saida)
    plt.close()


def selectkbest_features(df, k=10):
    from sklearn.feature_selection import SelectKBest, f_classif
    X = df.drop(columns=['rotulo','caminho']).values
    mapping = {c:i for i,c in enumerate(df['rotulo'].unique())}
    y = df['rotulo'].map(mapping).values
    skb = SelectKBest(f_classif, k=min(k, X.shape[1]))
    skb.fit(X, y)
    nomes = [f for f, s in zip(df.drop(columns=['rotulo','caminho']).columns, skb.get_support()) if s]
    return nomes


def importancia_random_forest(df):
    from sklearn.ensemble import RandomForestClassifier
    X = df.drop(columns=['rotulo','caminho']).values
    mapping = {c:i for i,c in enumerate(df['rotulo'].unique())}
    y = df['rotulo'].map(mapping).values
    rf = RandomForestClassifier(n_estimators=200, random_state=42)
    rf.fit(X, y)
    importancias = rf.feature_importances_
    nomes = df.drop(columns=['rotulo','caminho']).columns
    return pd.Series(importancias, index=nomes).sort_values(ascending=False)


def coeficientes_regressao_logistica(df):
    from sklearn.linear_model import LogisticRegression
    X = df.drop(columns=['rotulo','caminho']).values
    mapping = {c:i for i,c in enumerate(df['rotulo'].unique())}
    y = df['rotulo'].map(mapping).values
    lr = LogisticRegression(max_iter=1000)
    lr.fit(X, y)
    nomes = df.drop(columns=['rotulo','caminho']).columns
    coefs = np.mean(np.abs(lr.coef_), axis=0) if lr.coef_.ndim > 1 else lr.coef_
    return pd.Series(coefs, index=nomes).sort_values(ascending=False)


In [7]:
# Execução das análises EDA e salvamento de arquivos prontos para relatório

# Carregar dataset (arquivo em ../outputs/features.csv)
df_path = Path('..') / 'outputs' / 'features.csv'
df = carregar_features_csv(str(df_path))
print('Dataset carregado. Shape:', df.shape)

# Escolha de features promissoras (exemplo acadêmico)
promissoras = ['V_mean','frac_dark','glcm_contrast','lbp_var']
plot_boxplots_por_rotulo(df, promissoras, caminho_saida=str(Path('..')/'outputs'/'boxplots_promissoras_v2.png'))
print('Boxplots salvos: ../outputs/boxplots_promissoras_v2.png')

# PCA 2D
plot_pca_2d(df, caminho_saida=str(Path('..')/'outputs'/'pca_2d.png'))
print('PCA salvo: ../outputs/pca_2d.png')

# SelectKBest (top 10)
top_k = selectkbest_features(df, k=10)
print('SelectKBest top:', top_k)

# Importância por RandomForest
imp_rf = importancia_random_forest(df)
imp_rf.head(10).to_csv(str(Path('..')/'outputs'/'importancia_rf_top10.csv'))
print('Importância RF salva em: ../outputs/importancia_rf_top10.csv')

# Coeficientes da Regressão Logística
imp_lr = coeficientes_regressao_logistica(df)
imp_lr.head(10).to_csv(str(Path('..')/'outputs'/'coef_logreg_top10.csv'))
print('Coeficientes LogReg salvos em: ../outputs/coef_logreg_top10.csv')

# Salvar tabela final de features selecionadas (consenso simples)
consenso = list(dict.fromkeys(top_k + list(imp_rf.head(10).index) + list(imp_lr.head(10).index)))
pd.DataFrame({'feature': consenso}).to_csv(str(Path('..')/'outputs'/'features_selecionadas_consenso.csv'), index=False)
print('Features selecionadas (consenso) salvas em: ../outputs/features_selecionadas_consenso.csv')


Dataset carregado. Shape: (979, 13)
Boxplots salvos: ../outputs/boxplots_promissoras_v2.png
PCA salvo: ../outputs/pca_2d.png
SelectKBest top: ['H_mean', 'S_mean', 'V_mean', 'frac_dark', 'area', 'circularity', 'eccentricity', 'glcm_contrast', 'glcm_homogeneity', 'lbp_var']
Importância RF salva em: ../outputs/importancia_rf_top10.csv
Coeficientes LogReg salvos em: ../outputs/coef_logreg_top10.csv
Features selecionadas (consenso) salvas em: ../outputs/features_selecionadas_consenso.csv


c:\Users\cassi\Downloads\A1-VisaoComputacional\Sistema_de_classificacao_de_Imagens\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [15]:
# Extrair features e salvar em ../outputs/features.csv (pode demorar)
from notebooks.montar_dataset import montar_dataset
montar_dataset(str(Path('..')/'archive'), str(Path('..')/'outputs'/'features.csv'), max_por_classe=None)
print('Extração concluída. Arquivo gerado: ../outputs/features.csv')


Extraindo Withered: 100%|██████████| 54/54 [00:06<00:00,  8.02it/s]

Dataset salvo em ..\outputs\features.csv, shape=(979, 13)
Extração concluída. Arquivo gerado: ../outputs/features.csv


In [10]:
# Inspecionar colunas de features.csv (arquivo em ../outputs)
import pandas as pd
df_tmp = pd.read_csv(str(Path('..')/'outputs'/'features.csv'))
print('Colunas:', list(df_tmp.columns))
print(df_tmp.head(2).to_dict(orient='records'))


Colunas: ['H_mean', 'S_mean', 'V_mean', 'frac_dark', 'area', 'circularity', 'solidity', 'eccentricity', 'glcm_contrast', 'glcm_homogeneity', 'lbp_var', 'rotulo', 'caminho']
[{'H_mean': 18.493430425835047, 'S_mean': 89.89135891811212, 'V_mean': 154.6376896809064, 'frac_dark': 0.0, 'area': 44219.0, 'circularity': 0.2554182762408096, 'solidity': 0.9253353422478916, 'eccentricity': 0.8022554205186571, 'glcm_contrast': 0.0992907801418439, 'glcm_homogeneity': 0.950354609929078, 'lbp_var': 4.965481706142109, 'rotulo': 'Broken', 'caminho': 'archive\\Broken\\Broken_01.jpg'}, {'H_mean': 21.84431657800456, 'S_mean': 48.34187384648789, 'V_mean': 167.02228313972424, 'frac_dark': 0.0, 'area': 36844.0, 'circularity': 0.3603674291793043, 'solidity': 0.9420367671499068, 'eccentricity': 0.8915762452913231, 'glcm_contrast': 0.0598703724047017, 'glcm_homogeneity': 0.9700648137976492, 'lbp_var': 5.1466620051283005, 'rotulo': 'Broken', 'caminho': 'archive\\Broken\\Broken_02.jpg'}]


In [16]:
# Treinar modelos principais (RandomForest e LogisticRegression) e comparar com outros classificadores
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix, roc_auc_score
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

out_dir = Path('..') / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(str(out_dir/'features.csv'))
# preparar X,y
X = df.drop(columns=['rotulo','caminho']).values
cols = df.drop(columns=['rotulo','caminho']).columns.tolist()
mapping = {c:i for i,c in enumerate(sorted(df['rotulo'].unique()))}
y = df['rotulo'].map(mapping).values

# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
joblib.dump(scaler, str(out_dir/'scaler.joblib'))

# definicao de classificadores (inclui os 2 principais)
classifiers = {
    'RandomForest': RandomForestClassifier(n_estimators=200, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=2000, random_state=42),
    'SVC': SVC(probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'GradientBoosting': GradientBoostingClassifier(random_state=42),
    'GaussianNB': GaussianNB()
}

results = []
for name, clf in classifiers.items():
    print('Treinando', name)
    # usar X_train_s para métodos que se beneficiam do scaler; alguns (RF,GB) funcionam bem sem, mas usamos o escalador para padronizar
    try:
        clf.fit(X_train_s, y_train)
    except Exception:
        clf.fit(X_train, y_train)
    # previsões (tentar usar X_test_s, fallback para X_test)
    try:
        y_pred = clf.predict(X_test_s)
    except Exception:
        y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    prec = precision_score(y_test, y_pred, average='macro')
    rec = recall_score(y_test, y_pred, average='macro')
    # tentar ROC AUC (multiclasse OVR) se disponível
    auc = None
    try:
        if hasattr(clf, 'predict_proba'):
            probs = clf.predict_proba(X_test_s)
            auc = roc_auc_score(y_test, probs, multi_class='ovr')
        elif hasattr(clf, 'decision_function'):
            dec = clf.decision_function(X_test_s)
            auc = roc_auc_score(y_test, dec, multi_class='ovr')
    except Exception:
        auc = None
    results.append({'model': name, 'accuracy': acc, 'f1_macro': f1, 'precision_macro': prec, 'recall_macro': rec, 'roc_auc_ovr': auc})
    # salvar modelo e matrizes
    joblib.dump(clf, str(out_dir/f'model_{name}.joblib'))
    # salvar relatorio de classificacao
    with open(str(out_dir/f'class_report_{name}.txt'), 'w', encoding='utf-8') as fh:
        fh.write(classification_report(y_test, y_pred, target_names=sorted(mapping.keys())))
    # salvar matrix de confusao como figura
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion matrix - {name}')
    plt.xlabel('Predito')
    plt.ylabel('Verdadeiro')
    plt.tight_layout()
    plt.savefig(str(out_dir/f'cm_{name}.png'))
    plt.close()

# salvar resultados em csv
res_df = pd.DataFrame(results).sort_values('f1_macro', ascending=False)
res_df.to_csv(str(out_dir/'resultados_metricas_notebook.csv'), index=False)
print('Treino concluído. Métricas salvas em:', str(out_dir/'resultados_metricas_notebook.csv'))


Treinando RandomForest
Treinando LogisticRegression
Treinando SVC


c:\Users\cassi\Downloads\A1-VisaoComputacional\Sistema_de_classificacao_de_Imagens\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Treinando KNN
Treinando GradientBoosting
Treinando GaussianNB
Treino concluído. Métricas salvas em: ..\outputs\resultados_metricas_notebook.csv


In [17]:
# Resumo gráfico dos resultados gerados na célula anterior
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
out_dir = Path('..') / 'outputs'
df = pd.read_csv(str(out_dir/'resultados_metricas_notebook.csv'))
print(df)
# plot accuracies e f1
plt.figure(figsize=(8,4))
df.set_index('model')[['accuracy','f1_macro']].plot(kind='bar')
plt.title('Comparação: Accuracy e F1 (macro)')
plt.tight_layout()
plt.savefig(str(out_dir/'comparacao_accuracy_f1.png'))
plt.close()
print('Plots salvos em ../outputs/comparacao_accuracy_f1.png e matrizes de confusão individuais.')


                model  accuracy  f1_macro  precision_macro  recall_macro  \
0    GradientBoosting  0.647959  0.629458         0.649025      0.631967   
1        RandomForest  0.617347  0.586721         0.594069      0.593701   
2                 KNN  0.586735  0.566623         0.587417      0.572960   
3  LogisticRegression  0.596939  0.565515         0.561852      0.583767   
4                 SVC  0.576531  0.540764         0.546569      0.555552   
5          GaussianNB  0.484694  0.464510         0.494485      0.473467   

   roc_auc_ovr  
0     0.951069  
1     0.961079  
2     0.857751  
3     0.944564  
4     0.944138  
5     0.933410  
Plots salvos em ../outputs/comparacao_accuracy_f1.png e matrizes de confusão individuais.


<Figure size 960x480 with 0 Axes>